In [5]:
import os
import re
import json
import gc
import time
import hashlib
import numpy as np
import pandas as pd

In [6]:
TRAIN_IN = "/kaggle/input/datasets/zerotoinfinitya/train-with-label/train_with_Label.parquet"
VAL_IN = "/kaggle/input/datasets/zerotoinfinitya/val-with-label/val_with_Label.parquet"
TEST_IN = "/kaggle/input/datasets/zerotoinfinitya/test-with-label/test_with_Label.parquet"

TRAIN_OUT = "/kaggle/working/train_preprocessed.parquet"
VAL_OUT = "/kaggle/working/validation_preprocessed.parquet"
TEST_OUT = "/kaggle/working/test_preprocessed.parquet"

MAX_CHARS = 120000       
MAX_BYTES = 50000        

LABELS = ["access-control", "arithmetic", "other",
          "reentrancy", "safe", "unchecked-calls"]

In [7]:

OPCODES = {0x00: "STOP", 0x20: "KECCAK256"}

names = ["ADD", "MUL", "SUB", "DIV", "SDIV", "MOD", "SMOD", "ADDMOD",
         "MULMOD", "EXP", "SIGNEXTEND"]
for i in range(len(names)):
    OPCODES[0x01 + i] = names[i]

names = ["LT", "GT", "SLT", "SGT", "EQ", "ISZERO", "AND", "OR", "XOR",
         "NOT", "BYTE", "SHL", "SHR", "SAR"]
for i in range(len(names)):
    OPCODES[0x10 + i] = names[i]

names = ["ADDRESS", "BALANCE", "ORIGIN", "CALLER", "CALLVALUE",
         "CALLDATALOAD", "CALLDATASIZE", "CALLDATACOPY", "CODESIZE",
         "CODECOPY", "GASPRICE", "EXTCODESIZE", "EXTCODECOPY",
         "RETURNDATASIZE", "RETURNDATACOPY", "EXTCODEHASH"]
for i in range(len(names)):
    OPCODES[0x30 + i] = names[i]

names = ["BLOCKHASH", "COINBASE", "TIMESTAMP", "NUMBER", "PREVRANDAO",
         "GASLIMIT", "CHAINID", "SELFBALANCE", "BASEFEE", "BLOBHASH",
         "BLOBBASEFEE"]
for i in range(len(names)):
    OPCODES[0x40 + i] = names[i]

names = ["POP", "MLOAD", "MSTORE", "MSTORE8", "SLOAD", "SSTORE", "JUMP",
         "JUMPI", "PC", "MSIZE", "GAS", "JUMPDEST", "TLOAD", "TSTORE",
         "MCOPY", "PUSH0"]
for i in range(len(names)):
    OPCODES[0x50 + i] = names[i]

for i in range(32):
    OPCODES[0x60 + i] = "PUSH" + str(i + 1)
for i in range(16):
    OPCODES[0x80 + i] = "DUP" + str(i + 1)
    OPCODES[0x90 + i] = "SWAP" + str(i + 1)
for i in range(5):
    OPCODES[0xa0 + i] = "LOG" + str(i)

OPCODES[0xf0] = "CREATE"
OPCODES[0xf1] = "CALL"
OPCODES[0xf2] = "CALLCODE"
OPCODES[0xf3] = "RETURN"
OPCODES[0xf4] = "DELEGATECALL"
OPCODES[0xf5] = "CREATE2"
OPCODES[0xfa] = "STATICCALL"
OPCODES[0xfd] = "REVERT"
OPCODES[0xfe] = "INVALID"
OPCODES[0xff] = "SELFDESTRUCT"

In [8]:

def disassemble(hex_code):
    if not hex_code:
        return ""
    if hex_code[:2].lower() == "0x":
        hex_code = hex_code[2:]
    hex_code = re.sub(r"[^0-9a-fA-F]", "", hex_code)
    if len(hex_code) % 2 == 1:
        hex_code = hex_code[:-1]
    try:
        code = bytes.fromhex(hex_code)
    except ValueError:
        return ""

    out = []
    i = 0
    while i < len(code):
        op = code[i]
        name = OPCODES.get(op, "UNK")
        out.append(name)
        if 0x60 <= op <= 0x7f:
            i += 1 + (op - 0x5f)
        else:
            i += 1
    return " ".join(out)



disassembler self-test passed


In [9]:
def md5(text):
    return hashlib.md5(str(text or "").encode("utf-8", "ignore")).hexdigest()

In [13]:
def preprocess(in_path, out_path, name):
    df = pd.read_parquet(in_path)
    print("rows:", len(df))

    source = [("" if s is None else str(s)) for s in df["source_code"]]
    bytecode = [("" if b is None else str(b)) for b in df["bytecode"]]

    out = pd.DataFrame()
    out["address"] = df["address"].values
    out["source_capped"] = [s[:MAX_CHARS] for s in source]
    out["opcodes"] = [disassemble(b[:MAX_BYTES]) for b in bytecode]
    out["src_hash"] = [md5(s) for s in source]
    out["src_len"] = [len(s) for s in source]
    out["trunc_flag"] = (out["src_len"] > MAX_CHARS).astype(np.int8)

    out["bc_hash"] = [md5(b) for b in bytecode]
    out["bc_len"] = [len(b) for b in bytecode]
    out["bc_trunc_flag"] = (out["bc_len"] > MAX_BYTES).astype(np.int8)

    for c in LABELS:
        out[c] = df[c].values.astype(np.int8)

    out.to_parquet(out_path, index=False, compression="zstd")

    print("saved:", out_path)

    del df, out, source, bytecode
    gc.collect()



In [14]:
preprocess(TRAIN_IN, TRAIN_OUT, "train")

rows: 79641
saved: /kaggle/working/train_preprocessed.parquet


In [15]:
preprocess(VAL_IN, VAL_OUT, "validation")

rows: 10861
saved: /kaggle/working/validation_preprocessed.parquet


In [16]:
preprocess(TEST_IN, TEST_OUT, "test")

rows: 15972
saved: /kaggle/working/test_preprocessed.parquet
